# LSTM

In [ ]:
# =========================
# ⚠️ WARNING
# =========================
# This experiment uses a nested hyperparameter search.
# LSTM1 is controlled separately, while all other parameters
# are evaluated in combination.
#
# WARNING:
# This code will run a very large number of training iterations.
# Estimated runs can reach hundreds of models depending on combinations.
#
# Recommendation:
# - Start with ONE LSTM1 value first (32 OR 64 OR 128)
# - Do NOT run all configurations at once unless you have GPU support
# - Training time may take several hours on CPU
# =========================

import pandas as pd
import numpy as np
import random
import itertools

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt

# =========================
# 1. REPRODUCIBILITY
# =========================
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

# =========================
# 2. LOAD DATA
# =========================
data = pd.read_excel('DBD_Indonesia.xlsx')

data = data.drop(columns=[
    'dbd_fatalrate',
    'dbd_incidencerate',
    'dbd_deathtotal'
])

data = data.dropna()

# =========================
# 3. ENCODING
# =========================
for col in data.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])

# =========================
# 4. SCALE
# =========================
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X = data.drop(columns=['dbd_casetotal'])
y = data['dbd_casetotal']

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))

# =========================
# 5. SEQUENCE FUNCTION (PER PROVINCE)
# =========================
def create_sequences_grouped(df, time_steps=3):
    Xs, ys = [], []

    feature_cols = df.drop(columns=['dbd_casetotal']).columns

    for prov in df['Provinsi'].unique():
        prov_data = df[df['Provinsi'] == prov].sort_values('Tahun')

        X_p = prov_data[feature_cols].values
        y_p = prov_data['dbd_casetotal'].values.reshape(-1, 1)

        X_p_scaled = scaler_X.fit_transform(X_p)
        y_p_scaled = scaler_y.fit_transform(y_p)

        for i in range(len(prov_data) - time_steps):
            Xs.append(X_p_scaled[i:i+time_steps])
            ys.append(y_p_scaled[i+time_steps])

    return np.array(Xs), np.array(ys)

time_steps = 3
X_seq, y_seq = create_sequences_grouped(data, time_steps)

# =========================
# 6. SPLIT
# =========================
split = int(len(X_seq) * 0.8)

X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

# =========================
# 7. HYPERPARAMETERS
# =========================
lstm1_units = [32, 64, 128]

lstm2_units = [16, 32, 64]
dropout1_options = [0.2, 0.3, 0.4]
dropout2_options = [0.1, 0.2, 0.3]
dense_units_options = [8, 16, 32]
batch_size_options = [8, 16, 32]

epochs = 100

results = []

# =========================
# 8. OUTER LOOP: LSTM1 ONLY
# =========================
for l1 in lstm1_units:

    print(f"\n🔥 ================= LSTM1 = {l1} =================")

    # INNER GRID (all others)
    for l2, d1, d2, dn, bs in itertools.product(
        lstm2_units,
        dropout1_options,
        dropout2_options,
        dense_units_options,
        batch_size_options
    ):

        model = Sequential([
            LSTM(l1, return_sequences=True,
                 input_shape=(X_train.shape[1], X_train.shape[2])),
            Dropout(d1),

            LSTM(l2),
            Dropout(d2),

            Dense(dn, activation='relu'),
            Dense(1)
        ])

        model.compile(optimizer='adam', loss='mse')

        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        )

        model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=bs,
            validation_data=(X_test, y_test),
            callbacks=[early_stop],
            verbose=0
        )

        # prediction
        y_pred = model.predict(X_test)

        y_test_inv = scaler_y.inverse_transform(y_test)
        y_pred_inv = scaler_y.inverse_transform(y_pred)

        mse = mean_squared_error(y_test_inv, y_pred_inv)
        r2 = r2_score(y_test_inv, y_pred_inv)

        print(f"LSTM2={l2}, D1={d1}, D2={d2}, Dense={dn}, BS={bs} → MSE={mse:.2f}, R2={r2:.4f}")

        results.append({
            "lstm1": l1,
            "lstm2": l2,
            "dropout1": d1,
            "dropout2": d2,
            "dense": dn,
            "batch": bs,
            "mse": mse,
            "r2": r2
        })

# =========================
# 9. BEST MODEL
# =========================
results_df = pd.DataFrame(results)

print("\n🏆 TOP 5 MODELS:")
print(results_df.sort_values("mse").head(5))

